# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI, by de-duplicating from Andersen/GISAID, and relabeling sequences. 

In [1]:
# Housekeeping

import os
import pandas as pd
from collections import defaultdict
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [15]:
# Paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = home + "NCBI_Virus_Downloads/11-01-2021--04-14-2025/"
temp_files = home + "NCBI_Virus_Temp_Files/"
complete_files = home + "NCBI_Virus_Complete_Files/"
gisaid_andersen = home + "Complete_Combined_Files/11-01-2021--04-14-2025_all_genotypes/11-01-2021--04-14-2025_all_genotypes/"

# Day we're updating data
update_date = "04-14-2025"

os.chdir(downloads)

## De-Duplication

In [3]:
metadata = pd.read_csv("sequences.csv")

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.SRA_Accession, as_index=False).size()

print(metadata_counts)

metadata_counted = metadata.merge(metadata_counts, on="SRA_Accession")

# Only keep those with size=8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates

metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="first") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]

metadata_segments

     SRA_Accession  size
0      SRR23852495     8
1      SRR28752446     8
2      SRR28752447     8
3      SRR28752448     8
4      SRR28752449     8
...            ...   ...
4141   SRR33124773     8
4142   SRR33124774     8
4143   SRR33124775     8
4144   SRR33124776     8
4145   SRR33124777     8

[4146 rows x 2 columns]


,Accession,Organism_Name,GenBank_RefSeq,Assembly,SRA_Accession,Submitters,Organization,Org_location,Release_Date,Isolate,...,Segment,Geo_Location,USA,Host,Tissue_Specimen_Source,Collection_Date,BioSample,BioProject,GenBank_Title,size
0,PV571926,Influenza A virus,GenBank,GCA_049972485.1,SRR33124721,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-28,25-006338-001-original,...,1,USA: IN,IN,Meleagris gallopavo,oronasopharynx,2025-02-20,SAMN47941411,PRJNA980729,Influenza A virus (A/Turkey/IN/25-006338-001-o...,8
1,PV571927,Influenza A virus,GenBank,GCA_049972485.1,SRR33124721,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-28,25-006338-001-original,...,2,USA: IN,IN,Meleagris gallopavo,oronasopharynx,2025-02-20,SAMN47941411,PRJNA980729,Influenza A virus (A/Turkey/IN/25-006338-001-o...,8
2,PV571928,Influenza A virus,GenBank,GCA_049972485.1,SRR33124721,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-28,25-006338-001-original,...,3,USA: IN,IN,Meleagris gallopavo,oronasopharynx,2025-02-20,SAMN47941411,PRJNA980729,Influenza A virus (A/Turkey/IN/25-006338-001-o...,8
3,PV571929,Influenza A virus,GenBank,GCA_049972485.1,SRR33124721,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-28,25-006338-001-original,...,4,USA: IN,IN,Meleagris gallopavo,oronasopharynx,2025-02-20,SAMN47941411,PRJNA980729,Influenza A virus (A/Turkey/IN/25-006338-001-o...,8
4,PV571930,Influenza A virus,GenBank,GCA_049972485.1,SRR33124721,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-28,25-006338-001-original,...,5,USA: IN,IN,Meleagris gallopavo,oronasopharynx,2025-02-20,SAMN47941411,PRJNA980729,Influenza A virus (A/Turkey/IN/25-006338-001-o...,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37035,OQ565628,Influenza A virus,GenBank,GCA_039342815.1,SRR23852495,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,2023-03-08,VFAR-140,...,4,Peru,NaN,Pelecanus,lung,2022-12,SAMN33745292,PRJNA944237,Influenza A virus (A/Pelecanus/Peru/VFAR-140/2...,8
37036,OQ565629,Influenza A virus,GenBank,GCA_039342815.1,SRR23852495,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,2023-03-08,VFAR-140,...,5,Peru,NaN,Pelecanus,lung,2022-12,SAMN33745292,PRJNA944237,Influenza A virus (A/Pelecanus/Peru/VFAR-140/2...,8
37037,OQ565630,Influenza A virus,GenBank,GCA_039342815.1,SRR23852495,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,2023-03-08,VFAR-140,...,6,Peru,NaN,Pelecanus,lung,2022-12,SAMN33745292,PRJNA944237,Influenza A virus (A/Pelecanus/Peru/VFAR-140/2...,8
37038,OQ565631,Influenza A virus,GenBank,GCA_039342815.1,SRR23852495,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,2023-03-08,VFAR-140,...,7,Peru,NaN,Pelecanus,lung,2022-12,SAMN33745292,PRJNA944237,Influenza A virus (A/Pelecanus/Peru/VFAR-140/2...,8


In [4]:
# De-duplicate from Andersen/GISAID using isolate

# If even one isolate in this list exists in the NCBI Virus dataframe, remove it from NCBI Virus dataframe
gisaid_andersen_isolates = []
# Grab files
for dirpath, dirs, files in os.walk(gisaid_andersen):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name) # Convert fasta file to dataframe
            isolates = fasta_file["Isolate_Id"].apply(lambda x: x.split("/")[3])
            for index, value in isolates.items():
                gisaid_andersen_isolates.append(value)
         
    break 

gisaid_andersen_isolates = list(set(gisaid_andersen_isolates))

print(gisaid_andersen_isolates)

['W22-199D', '22-026118-002', '24-035157-003', '026360-005', '22-036768-001', '22-029901-002-R', '031285-013', 'FAV-0415-7', '24_034221-001', '036403-004', '006223-001', '22-017940-010', '22-019668-001-original', 'W241070094-25', '22-018423-002', '007113-001', '24-020285-004', '23-037672-002', '033876-002', '22-036105-005', '002281-001', '22-015021-001', '036563-004', '22-024871-017', '004074-009', '24_034194-001', '031666-028', '24_018154-001', 'AIVPHL-1151', '22-028884-003', '22-008483-001', '24-008357-001', '002143-001', '038278-001', 'AIVPHL-23', '038947-002', '078707-24-10', 'FAV-0833-30', '24-009315-003-original', '037207-001', '24_034232-001', '035654-001', 'USDA-011034-001', '22-017507-007', '23-009211-001', '23-034904-001', 'B24OSU-472', 'AIVPHL-1443', 'FAV-0300-05', '22-012047-001', 'FAV-1010-3', '23-035121-001-original', '002467-001', '015884-014', '22-039352-016', '22-014661-002', '23-034924-012-original', '24_009110-017', '22-014175-003', '007839-001', '032038-001', '23-04

In [5]:
# Remove duplicates from GISAID/Andersen

print(len(metadata_segments))

for index, value in metadata_segments["Isolate"].items():
    if value in gisaid_andersen_isolates:
        metadata_segments = metadata_segments.drop(index)

print(len(metadata_segments))

metadata_segments



29400
29248


,Accession,Organism_Name,GenBank_RefSeq,Assembly,SRA_Accession,Submitters,Organization,Org_location,Release_Date,Isolate,...,Segment,Geo_Location,USA,Host,Tissue_Specimen_Source,Collection_Date,BioSample,BioProject,GenBank_Title,size
0,PV571926,Influenza A virus,GenBank,GCA_049972485.1,SRR33124721,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-28,25-006338-001-original,...,1,USA: IN,IN,Meleagris gallopavo,oronasopharynx,2025-02-20,SAMN47941411,PRJNA980729,Influenza A virus (A/Turkey/IN/25-006338-001-o...,8
1,PV571927,Influenza A virus,GenBank,GCA_049972485.1,SRR33124721,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-28,25-006338-001-original,...,2,USA: IN,IN,Meleagris gallopavo,oronasopharynx,2025-02-20,SAMN47941411,PRJNA980729,Influenza A virus (A/Turkey/IN/25-006338-001-o...,8
2,PV571928,Influenza A virus,GenBank,GCA_049972485.1,SRR33124721,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-28,25-006338-001-original,...,3,USA: IN,IN,Meleagris gallopavo,oronasopharynx,2025-02-20,SAMN47941411,PRJNA980729,Influenza A virus (A/Turkey/IN/25-006338-001-o...,8
3,PV571929,Influenza A virus,GenBank,GCA_049972485.1,SRR33124721,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-28,25-006338-001-original,...,4,USA: IN,IN,Meleagris gallopavo,oronasopharynx,2025-02-20,SAMN47941411,PRJNA980729,Influenza A virus (A/Turkey/IN/25-006338-001-o...,8
4,PV571930,Influenza A virus,GenBank,GCA_049972485.1,SRR33124721,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-28,25-006338-001-original,...,5,USA: IN,IN,Meleagris gallopavo,oronasopharynx,2025-02-20,SAMN47941411,PRJNA980729,Influenza A virus (A/Turkey/IN/25-006338-001-o...,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37035,OQ565628,Influenza A virus,GenBank,GCA_039342815.1,SRR23852495,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,2023-03-08,VFAR-140,...,4,Peru,NaN,Pelecanus,lung,2022-12,SAMN33745292,PRJNA944237,Influenza A virus (A/Pelecanus/Peru/VFAR-140/2...,8
37036,OQ565629,Influenza A virus,GenBank,GCA_039342815.1,SRR23852495,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,2023-03-08,VFAR-140,...,5,Peru,NaN,Pelecanus,lung,2022-12,SAMN33745292,PRJNA944237,Influenza A virus (A/Pelecanus/Peru/VFAR-140/2...,8
37037,OQ565630,Influenza A virus,GenBank,GCA_039342815.1,SRR23852495,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,2023-03-08,VFAR-140,...,6,Peru,NaN,Pelecanus,lung,2022-12,SAMN33745292,PRJNA944237,Influenza A virus (A/Pelecanus/Peru/VFAR-140/2...,8
37038,OQ565631,Influenza A virus,GenBank,GCA_039342815.1,SRR23852495,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,2023-03-08,VFAR-140,...,7,Peru,NaN,Pelecanus,lung,2022-12,SAMN33745292,PRJNA944237,Influenza A virus (A/Pelecanus/Peru/VFAR-140/2...,8


In [6]:
# NCBI Virus Naming Convention Example:
# Influenza A virus |USA: IN|25-006338-001-original|H5N1|2025-02-20|Meleagris gallopavo|GenBank|SRR33124721|SAMN47941411|PRJNA980729|Influenza A virus (A/Turkey/IN/25-006338-001-original/2025(H5N1)) segment 1 polymerase PB2 (PB2) gene, complete cds
# Organism_Name | Geo_Location | Isolate | Genotype | Collection_Date | Host | GenBank_RefSeq | SRA_Accession | BioSample | BioProject | GenBank_Title


# print(metadata_segments["GenBank_Title"].iloc[0])

# Get sequences and headers together
headers = []
isolates = []
sras = []
headers_seqs = {}
with open("sequences.fasta") as f:
    lines = f.readlines()
    for num, line in enumerate(lines):
        if line[0] == ">": # If this is a header
            if line.strip() not in headers: # And is not a header we've seen before
                header = line.strip() 
                # print(header)
                split_header = header.split("|")
                # print(split_header)
                headers.append(header) 
                if num < len(lines): # If we're not at the last line
                    # for i, l in enumerate(lines[num + 1:]):
                    i = num
                    sequence = ""
                    # print(lines[i])
                    # print(lines[i + 1])
                    while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                        sequence = sequence + lines[i + 1].strip()
                        i += 1
                    headers_seqs[header] = sequence # Add next lines to sequences
                # name = split_header[-1] # Last part gives GenBank name, with segment and isolate
                isolate = split_header[2]
                isolates.append(isolate)
                sra = split_header[7]
                sras.append(sra)
    f.close()



# print(mask_all.all(axis=1))

In [7]:
# Double-check de-duplication
mask = metadata_segments["Isolate"].isin(isolates) # Check if isolate is in fasta
mask_sra = metadata_segments["SRA_Accession"].isin(sras) # Check if sra is in fasta

metadata_masked = metadata_segments[mask] # Some do not have isolates, so check sra too
metadata_masked_sra = metadata_segments[mask_sra]

merged_metadata = metadata_masked.merge(metadata_masked_sra, how="outer")
merged_metadata = merged_metadata.drop_duplicates(keep="first")

print(len(headers_seqs))
print(len(merged_metadata))

75007
29248


In [8]:
# Add sequences to a dataframe 
headers_seqs_df = pd.DataFrame.from_dict(headers_seqs, orient="index")
headers_seqs_df = headers_seqs_df.reset_index()
headers_seqs_df["GenBank_Title"] = headers_seqs_df["index"].apply(lambda x: x[1:].split("|")[-1])
headers_seqs_df = headers_seqs_df.rename(columns={0: "Sequence", "index": "Header"})

# Add sequences to the dataframe
metadata_seqs = merged_metadata.merge(headers_seqs_df, on="GenBank_Title", how="left")

print(metadata_seqs)


      Accession      Organism_Name GenBank_RefSeq         Assembly  \
0      OQ565625  Influenza A virus        GenBank  GCA_039342815.1   
1      OQ565626  Influenza A virus        GenBank  GCA_039342815.1   
2      OQ565627  Influenza A virus        GenBank  GCA_039342815.1   
3      OQ565628  Influenza A virus        GenBank  GCA_039342815.1   
4      OQ565629  Influenza A virus        GenBank  GCA_039342815.1   
...         ...                ...            ...              ...   
29251  PV573369  Influenza A virus        GenBank  GCA_049972655.1   
29252  PV573370  Influenza A virus        GenBank  GCA_049972655.1   
29253  PV573371  Influenza A virus        GenBank  GCA_049972655.1   
29254  PV573372  Influenza A virus        GenBank  GCA_049972655.1   
29255  PV573373  Influenza A virus        GenBank  GCA_049972655.1   

      SRA_Accession                                         Submitters  \
0       SRR23852495  Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...   
1       SRR

In [24]:
# Create FASTA files per Header

metadata_seqs["Partial_Header"] = metadata_seqs["Header"].apply(lambda x: "|".join(x.split("|")[:-1]))

unique_partial_headers = []
df_list = []
# Each header should repeat 8 times
for partial_header in metadata_seqs["Partial_Header"].values:
    if partial_header not in unique_partial_headers: # Only use the first occurence
        unique_partial_headers.append(partial_header)

print(len(unique_partial_headers)) # Sanity check

3657


In [25]:
for partial_header in unique_partial_headers:
    # Get smaller dataframe
    df = metadata_seqs[metadata_seqs["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    df = df[["Header", "Sequence"]]
    df = df.rename(columns={"Header": "full_header", "Sequence": "sequence"}) # For utils function
    # Make sure there are 8 segments
    if len(df) == 8:
        df_list.append(df)

In [49]:
# Make fasta files. Finally

os.chdir(temp_files)

for df in df_list:
    print(df)
    file_name = df["full_header"].apply(lambda x: "".join(x.split("|")[:-2])).values[0] + "temp.fasta"
    # Forbidden characters
    for c in [">", "/", "|", " ", ":", ",", "(", ")"]:
        file_name = file_name.replace(c, "")
    print(file_name)
    df_to_fasta(df, file_name, temp_files)

                                         full_header  \
0  >Influenza A virus |Peru|VFAR-140|H5N1|2022-12...   
1  >Influenza A virus |Peru|VFAR-140|H5N1|2022-12...   
2  >Influenza A virus |Peru|VFAR-140|H5N1|2022-12...   
3  >Influenza A virus |Peru|VFAR-140|H5N1|2022-12...   
4  >Influenza A virus |Peru|VFAR-140|H5N1|2022-12...   
5  >Influenza A virus |Peru|VFAR-140|H5N1|2022-12...   
6  >Influenza A virus |Peru|VFAR-140|H5N1|2022-12...   
7  >Influenza A virus |Peru|VFAR-140|H5N1|2022-12...   

                                            sequence  
0  AAAAGCARGTCAARTATATTCAATATGGAGAGAAKAAAAGARCTAA...  
1  AGCGAAAGCAAACAAACCATTTGAATGGATGTCAATCCGAYTTTRC...  
2  CAAATGCNNKTRCTGATTYAAAATGGAAGACTTTGTGCGACAATGC...  
3  GTCAAAATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTG...  
4  TAGATAATCACTCACTGAGTGRYATSCACATCATGGCRTMYCARGR...  
5  CCATTGGATCARTCTGTATGGTAATTGGGATAGTCAGYTTGATGCT...  
6  TTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCT...  
7  GCTACAGCWGGGTGACAAAAACATAATGGATTCCAACACTGTGTCA...  
